# C07d — Hosted GPTZero: all 54 OG vs NS10 pairs

Evaluate the fixed NS10 endpoint using the official GPTZero v2 prediction API. No local GPU or detector training is needed. **API access is required**; a web-app plan does not necessarily provide API access. This notebook does not purchase anything. API requests send full document text to GPTZero and may consume paid quota.

Primary ranking: `class_probabilities.ai` (AI-only). Prespecified secondary ranking: `ai + mixed` (any AI involvement). Both are reported, never selected by which makes the attack look better. No legacy probability field is silently substituted. Calibration uses original RAID validation humans; evaluation uses original test humans reweighted to the same 54-case genre mix. No training documents are sent.

The prediction API exposes no retraining operation: this notebook reports **native transfer only**, not adaptive camouflage. A post-hoc wrapper would be a different detector. API updates cannot be fully controlled without a supported version pin; request/response times and returned version metadata are saved.

1. Run setup and inspect request/word counts.
2. Supply `GPTZERO_API_KEY` through the environment or hidden prompt.
3. Set `RUN_API=True` in the submission cell to run the batch.
4. Run analysis. Every successful response is cached; reruns reuse it.

Official references (checked 2026-09-23): https://gptzero.me/developers ; https://support.gptzero.me/articles/8947054519-how-do-i-use-and-interpret-the-results-from-your-api ; https://support.gptzero.me/articles/5840144813-how-can-i-get-the-api-and-request-code-samples


In [1]:
from pathlib import Path
import os, json, hashlib, tempfile, time, getpass
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from IPython.display import display
ROOT_OVERRIDE=None
ROOT=Path(ROOT_OVERRIDE) if ROOT_OVERRIDE else next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'results/causal/C02d2_fz118_ghostbuster_raid').exists()),None)
assert ROOT is not None,'Set ROOT_OVERRIDE to reproduction/.'
BASE=ROOT/'results/causal/C02d2_fz118_ghostbuster_raid'
C05=ROOT/'results/causal/C05_Ghostbuster_RAID_fixed54'
NSDIR=ROOT/'results/causal/C06b_NEW_steps5_10_v1'
OUT=ROOT/'results/causal/C07d_GPTZero_OG_NS10_v1'
for sub in ['', 'responses','errors','figures']:(OUT/sub).mkdir(parents=True,exist_ok=True)
ENDPOINT='https://api.gptzero.me/v2/predict/text'
API_VERSION=None # Optional supported version string from your GPTZero account/docs.
SEED=20260923
SPEC={'endpoint':ENDPOINT,'requested_version':API_VERSION,'input':'full_document','schema':1}
def sha(text):return hashlib.sha256(str(text).encode()).hexdigest()
def atomic_json(path,value):
    fd,tmp=tempfile.mkstemp(dir=path.parent,suffix='.tmp')
    try:
        with os.fdopen(fd,'w') as f:json.dump(value,f,indent=2,sort_keys=True,allow_nan=False);f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)
def now():return datetime.now(timezone.utc).isoformat()
def cache_path(text):return OUT/'responses'/(sha(json.dumps(SPEC,sort_keys=True)+sha(text))+'.json')
def parse_prediction(body):
    docs=body.get('documents')
    if not isinstance(docs,list) or len(docs)!=1:raise ValueError('Expected one entry in documents; inspect saved raw response.')
    doc=docs[0];p=doc.get('class_probabilities')
    if not isinstance(p,dict) or not all(k in p for k in ['ai','human','mixed']):raise ValueError('Missing class_probabilities ai/human/mixed; raw response preserved.')
    vals={k:float(p[k]) for k in ['ai','human','mixed']}
    if not all(np.isfinite(v) and 0<=v<=1 for v in vals.values()) or not np.isclose(sum(vals.values()),1,atol=.005):raise ValueError('Invalid class probabilities; inspect response.')
    return {'AI_only':vals['ai'],'AI_or_mixed':vals['ai']+vals['mixed'],'human_probability':vals['human'],'classification':doc.get('document_classification'),'returned_version':str(doc.get('version',body.get('version','unreported')))}
print('Output:',OUT)


Output: ../results/causal/C07d_GPTZero_OG_NS10_v1


In [2]:
meta=pd.read_parquet(BASE/'fz118_paired_features.parquet')
corpus=pd.read_parquet(BASE/'paired_corpus.parquet')
assert meta.source_id.is_unique and corpus.source_id.is_unique
raid=meta.loc[meta.dataset.eq('raid'),['source_id','split','label','domain']].merge(corpus[['source_id','text']],on='source_id',validate='one_to_one')
keys=pd.read_parquet(C05/'raid_all_features.parquet',columns=['doc_key']).doc_key
assert keys.is_unique and set(keys)==set('raid::'+raid.source_id.astype(str))
assert raid.text.notna().all()
raid['pair_group']=raid.source_id.str.split('__',n=1).str[-1]
assert raid.groupby('pair_group').split.nunique().eq(1).all()
raid['text_sha']=raid.text.map(sha)
assert raid.groupby('text_sha').split.nunique().eq(1).all()
ns=pd.read_csv(NSDIR/'NS10_texts.csv',keep_default_na=False).sort_values('case_id').reset_index(drop=True)
ns['case_id']=ns.case_id.astype(str)
assert len(ns)==54 and ns.case_id.is_unique and ns.source_id.is_unique
assert ns.text.map(sha).eq(ns.text_sha).all()
r=raid.set_index('source_id')
assert r.loc[ns.source_id,'split'].eq('test').all() and r.loc[ns.source_id,'label'].eq(1).all()
assert r.loc[ns.source_id,'domain'].tolist()==ns.domain.tolist()
og=ns[['case_id','source_id','domain']].copy();og['text']=og.source_id.map(r.text)
mix=og.domain.value_counts(normalize=True)
hum=raid.loc[raid.label.eq(0)&raid.split.isin(['validation','test'])&raid.domain.isin(mix.index)].copy()
hum['case_id']=hum.source_id.astype(str);hum['arm']='Human'
og['arm']='OG';og['split']='test';ns['arm']='NS10';ns['split']='test'
columns=['case_id','source_id','domain','arm','split','text']
docs=pd.concat([og[columns],ns[columns],hum[columns]],ignore_index=True)
assert docs.text.str.strip().ne('').all()
docs['text_sha']=docs.text.map(sha)
manifest=docs.drop(columns='text').to_dict('records')
protocol={'request':SPEC,'documents':manifest}
pp=OUT/'protocol.json'
if pp.exists():assert json.loads(pp.read_text())==protocol,'Inputs changed; use a new OUT for a new experiment.'
else:atomic_json(pp,protocol)
docs.drop(columns='text').to_csv(OUT/'document_manifest.csv',index=False)
# Fixed shuffled order interleaves arms to reduce systematic time/version confounding.
unique=list(dict.fromkeys(docs.text));order=np.random.default_rng(SEED).permutation(len(unique));ordered=[unique[i] for i in order]
pending=[t for t in ordered if not cache_path(t).exists()]
display(docs.groupby(['arm','split']).size().rename('documents').to_frame())
print('Unique requests total:',len(unique),'| cached:',len(unique)-len(pending),'| remaining:',len(pending))
print('Approximate remaining whitespace words:',sum(len(t.split()) for t in pending))
print('Includes validation/test humans required for ROC and calibration. No RAID training texts submitted.')
del meta,corpus,r,keys


documents
arm   split                
Human test              128
      validation        125
NS10  test               54
OG    test               54

Unique requests total: 361 | cached: 0 | remaining: 361
Approximate remaining whitespace words: 95142
Includes validation/test humans required for ROC and calibration. No RAID training texts submitted.


In [ ]:
RUN_API=True # Set True after inspecting the quota estimate above.
if not RUN_API:
    print('No requests sent. Set RUN_API=True when ready.')
else:
    pending=[t for t in ordered if not cache_path(t).exists()]
    if pending:
        api_key=os.environ.get('GPTZERO_API_KEY') or getpass.getpass('GPTZero API key (hidden): ')
        assert api_key.strip(),'API key required.'
        session=requests.Session()
        session.headers.update({'x-api-key':api_key.strip(),'Accept':'application/json','Content-Type':'application/json'})
        try:
            for i,text in enumerate(pending,1):
                payload={'document':text}
                if API_VERSION is not None:payload['version']=API_VERSION
                # No automatic retry after a timeout: the server may already have billed it.
                started=now()
                response=session.post(ENDPOINT,json=payload,timeout=(15,180))
                try:body=response.json()
                except ValueError:body={'non_json_response':response.text[:5000]}
                record={'text_sha':sha(text),'request_spec':SPEC,'started_at':started,'completed_at':now(),'status_code':response.status_code,'request_id':response.headers.get('x-request-id'),'response':body}
                if not response.ok:
                    atomic_json(OUT/'errors'/(sha(text)+'.json'),record)
                    raise RuntimeError(f'GPTZero HTTP {response.status_code}. Error saved; batch stopped. Check API access/quota or input limits; no truncation or automatic retries applied.')
                atomic_json(cache_path(text),record) # Persist before parsing; schema errors do not cost another call.
                parse_prediction(body)
                print(f'{i}/{len(pending)} saved',flush=True)
                time.sleep(.5)
        finally:
            session.headers.pop('x-api-key',None);session.close();del api_key
    else:print('All responses already cached.')


KeyboardInterrupt: Interrupted by user

In [ ]:
missing=[t for t in ordered if not cache_path(t).exists()]
assert not missing,f'{len(missing)} requests remain. Complete the API cell; no silent case exclusions.'
rows=[]
for row in docs.to_dict('records'):
    raw=json.loads(cache_path(row['text']).read_text());pred=parse_prediction(raw['response'])
    rows.append({**{k:v for k,v in row.items() if k!='text'},**pred,'requested_at':raw['started_at']})
scores=pd.DataFrame(rows);scores.to_csv(OUT/'scores.csv',index=False)
display(scores.groupby(['returned_version','arm']).size().rename('documents').to_frame())
print('Request range:',scores.requested_at.min(),'to',scores.requested_at.max())
if scores.returned_version.nunique()>1:print('WARNING: multiple returned versions; interpret pooled results as mixed-service-version evaluation.')
if scores.returned_version.eq('unreported').any():print('Service did not report a version for some documents; timestamps are retained.')


AssertionError: 361 requests remain. Complete the API cell; no silent case exclusions.

In [ ]:
def weights(frame):
    counts=frame.domain.value_counts();assert set(mix.index)<=set(counts.index)
    w=np.array([mix[d]/counts[d] for d in frame.domain]);assert np.isclose(w.sum(),1);return w
human=scores.loc[scores.arm.eq('Human')&scores.split.eq('test')]
cal=scores.loc[scores.arm.eq('Human')&scores.split.eq('validation')]
w=weights(human);cw=weights(cal)
ids=og.case_id.tolist();summary=[];operating=[];stairs=[];changes=[]
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for ax,metric in zip(axes,['AI_only','AI_or_mixed']):
    h=human[metric].to_numpy();c=cal[metric].to_numpy();auc_by_case={}
    for arm,color in [('OG','#858C96'),('NS10','#8B5FBF')]:
        g=scores.loc[scores.arm.eq(arm)].set_index('case_id').loc[ids];a=g[metric].to_numpy()
        per_case=((a[:,None]>h).astype(float)+.5*(a[:,None]==h))@w;auc=per_case.mean();auc_by_case[arm]=per_case
        fp,tp,th=roc_curve(np.r_[np.zeros(len(h)),np.ones(len(a))],np.r_[h,a],sample_weight=np.r_[w,np.ones(len(a))/len(a)],drop_intermediate=False)
        ax.step(100*fp,100*tp,where='post',color=color,label=f'{arm}: AUROC {auc:.3f}')
        summary.append(dict(metric=metric,arm=arm,cases=len(a),AUROC=auc,mean_score=a.mean()))
        stairs.extend(dict(metric=metric,arm=arm,FPR=float(x),TPR=float(y),threshold=float(t)) for x,y,t in zip(fp,tp,th))
        for target in [.01,.025,.05,.10,.20,.50]:
            cut=min(t for t in np.r_[np.inf,np.unique(c)] if cw[c>=t].sum()<=target+1e-12)
            operating.append(dict(metric=metric,arm=arm,nominal_FPR=target,threshold=cut,actual_test_FPR=w[h>=cut].sum(),TPR=(a>=cut).mean()))
    delta=auc_by_case['NS10']-auc_by_case['OG'];rng=np.random.default_rng(SEED)
    boot=delta[rng.integers(0,len(ids),(5000,len(ids)))].mean(axis=1);lo,hi=np.quantile(boot,[.025,.975])
    changes.append(dict(metric=metric,comparison='NS10 minus OG',AUROC_delta=delta.mean(),ci_low=lo,ci_high=hi))
    ax.plot([0,100],[0,100],':',color='gray');ax.set(xlim=(0,100),ylim=(0,100),xlabel='Test-human FPR (%)',ylabel='TPR (%)',title='Primary: P(AI-only)' if metric=='AI_only' else 'Secondary: P(AI-only)+P(mixed)');ax.grid(alpha=.2);ax.legend(loc='lower right')
fig.suptitle('C07d — hosted GPTZero | 54 matched OG/NS10 pairs')
for ext in ['png','pdf']:fig.savefig(OUT/'figures'/f'GPTZero_OG_NS10_54.{ext}',dpi=220)
plt.show();plt.close(fig)
for name,data in [('summary',summary),('operating_points',operating),('staircases',stairs),('paired_changes',changes)]:
    frame=pd.DataFrame(data);frame.to_csv(OUT/f'{name}.csv',index=False)
    if name!='staircases':display(frame.round(4))
print('Intervals resample paired AI cases only; humans and API outputs fixed. No service/version uncertainty included.')
print('Saved:',OUT)
